# Statistical Analysis & Quality Control Pipeline
**Objective:** Process numerical datasets, evaluate location/dispersion parameters, assess capability indices, and generate control charts.

In [ ]:
# Phase 1: Environment Setup & Data Ingestion
import numpy as np
import scipy.stats as sta
from scipy.integrate import quad
import matplotlib.pyplot as plt
import time as t

# Load dataset from text file
data_path = 'data.txt'
values = np.loadtxt(data_path)  # Load sequence
n = len(values)

# Reshape into sample matrix (5 rows per batch) if running SPC
rows, columns = 5, n // 5
sample_table = np.reshape(values[:rows*columns], (rows, columns), order='F')  # Fortran order column formatting
print(f"Loaded {n} records into a matrix of shape {sample_table.shape}")

In [ ]:
# Phase 2: Location & Dispersion Parameter Evaluation
mean_val = np.mean(values)  # Arithmetic mean
median_val = np.median(values)  # Median
mode_res = sta.mode(values, axis=None)  # Modal value
hmean_val = sta.hmean(np.abs(values))  # Harmonic mean
gmean_val = sta.gmean(np.abs(values))  # Geometric mean
std_val = np.std(values, ddof=1)  # Sample standard deviation (n-1)
skew_val = sta.skew(values)  # Exact distribution skewness

print(f"Mean: {mean_val:.4f} | Median: {median_val:.4f} | Mode: {mode_res.mode}")
print(f"Std Dev: {std_val:.4f} | Skewness: {skew_val:.4f} ({'Left' if skew_val > 0 else 'Right'}-skewed)")

In [ ]:
# Phase 3: Machine Capability Analysis (Cm & Cmk)
setpoint = 50.0  # Target setpoint value
To, Tu = 5.0, -5.0  # Upper/Lower tolerance specification limits
T = To - Tu  # Total tolerance range

Cm = T / (6 * std_val)  # Machine capability index[cite: 1]
UCL_spec, LCL_spec = setpoint + To, setpoint + Tu  # Tolerance limits[cite: 1]
delta_k = min(UCL_spec - mean_val, mean_val - LCL_spec)  # Critical deviation distance[cite: 1]
Cmk = delta_k / (3 * std_val)  # Machine capability characteristic value[cite: 1]

print(f"Capability Index (Cm): {Cm:.4f}")
print(f"Critical Capability (Cmk): {Cmk:.4f} (Status: {'PASS' if Cmk >= 1.67 else 'FAIL'})")[cite: 1]

In [ ]:
# Phase 4: Quality Control Chart Simulation (x-bar & s Charts)
A3, B4 = 1.152, 1.669  # Factors for sample size n=5[cite: 1]
mw = [np.mean(sample_table[:, i]) for i in range(columns)]  # Batch means[cite: 1]
staw = [np.std(sample_table[:, i], ddof=1) for i in range(columns)]  # Batch std dev[cite: 1]

mmw, mws = np.mean(mw), np.mean(staw)  # Global sample averages[cite: 1]
UCLm, LCLm = mmw + A3 * mws, mmw - A3 * mws  # Control limits for means[cite: 1]
UCLs = B4 * mws  # Control limit for standard deviations[cite: 1]

fig, ax = plt.subplots(2, 1, figsize=(8, 6))
h = np.arange(1, columns + 1)
ax[0].plot([1, columns], [UCLm, UCLm], 'r--', [1, columns], [mmw, mmw], 'g-', [1, columns], [LCLm, LCLm], 'r--')[cite: 1]
ax[0].plot(h, mw, 'bx-')[cite: 1]
ax[0].set_title("Mean Value Chart (x-bar)")[cite: 1]

ax[1].plot([1, columns], [UCLs, UCLs], 'r--')[cite: 1]
ax[1].plot(h, staw, 'gx-')[cite: 1]
ax[1].set_title("Standard Deviation Chart (s)")[cite: 1]
fig.tight_layout()[cite: 1]
plt.show()

In [ ]:
# Phase 5: Linear Regression Analysis
# Example input arrays X (independent) and Y (dependent)[cite: 1]
X = np.array([46, 53, 29, 61, 36, 39, 47, 49, 52, 38, 55, 32, 57, 54, 44])[cite: 1]
Y = np.array([12, 15, 7, 17, 10, 11, 11, 12, 14, 9, 16, 8, 18, 14, 12])[cite: 1]

m_slope, a_intercept, r_val, p_val, std_err = sta.linregress(X, Y)  # Fit linear model[cite: 1]
print(f"Model: y = {m_slope:.4f}x + {a_intercept:.4f} | Correlation r: {r_val:.4f}")[cite: 1]

plt.figure(figsize=(6, 4))
plt.plot(X, Y, 'rx', label='Data')[cite: 1]
plt.plot(X, m_slope * X + a_intercept, 'b-', label='Fit')[cite: 1]
plt.xlabel("Influencing Variable X")[cite: 1]
plt.ylabel("Outcome Variable Y")[cite: 1]
plt.legend()[cite: 1]
plt.show()